In [3]:
! pip install mysql-connector-python

In [17]:
import mysql.connector
import pandas

In [18]:
# Connect to MySQL server
conn = mysql.connector.connect(
    host = '127.0.0.1',
    port = 3306,
    user = 'root',
    password = '401721',
)
cursor = conn.cursor()

In [21]:
# Correct SQL syntax
cursor.execute("CREATE DATABASE IF NOT EXISTS pizza_db")

print("✅ Database 'pizza_db' created or already exists.")

# Switch to that database
cursor.execute("USE pizza_db")
print("✅ Switched to database 'pizza_db' successfully.")

✅ Database 'pizza_db' created or already exists.
✅ Switched to database 'pizza_db' successfully.


etl_job(Using mysql_connector)

In [ ]:
import mysql.connector
import pandas as pd
import logging
import os
import sys

# ==============================
# DATABASE CONNECTION (your code)
# ==============================
try:
    conn = mysql.connector.connect(
        host='127.0.0.1',
        port=3306,
        user='root',
        password='401721'
    )
    cursor = conn.cursor()

    cursor.execute("CREATE DATABASE IF NOT EXISTS pizza_db")
    print("✅ Database 'pizza_db' created or already exists.")

    cursor.execute("USE pizza_db")
    print("✅ Switched to database 'pizza_db' successfully.")

except Exception as e:
    print("❌ Database connection error:", e)
    sys.exit(1)

# ==============================
# CONFIGURATION
# ==============================
CSV_FILE_PATH = 'Dataset/pizza_sales.csv'  # your CSV file path
TABLE_NAME = 'pizza_sales'

os.makedirs('log', exist_ok=True)
logging.basicConfig(
    filename='log/etl_pizza.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ==============================
# EXTRACT
# ==============================
def extract(csv_path):
    try:
        logging.info(f"Extracting data from {csv_path}")
        df = pd.read_csv(csv_path)
        logging.info(f"Extraction complete. Rows: {len(df)} Columns: {list(df.columns)}")
        return df
    except Exception as e:
        logging.error(f"Error during extraction: {e}")
        sys.exit(1)

# ==============================
# TRANSFORM
# ==============================
def transform(df):
    try:
        logging.info("Starting transformation")
        df.columns = df.columns.str.lower().str.replace(' ', '_')
        df.drop_duplicates(inplace=True)
        df.fillna(value=None, inplace=True)
        logging.info("Transformation complete")
        return df
    except Exception as e:
        logging.error(f"Error during transformation: {e}")
        sys.exit(1)

# ==============================
# LOAD
# ==============================
def load(df, cursor, conn, table_name):
    try:
        logging.info(f"Creating table {table_name} if not exists")

        # Dynamically build CREATE TABLE query (all TEXT columns)
        cols = ", ".join([f"`{col}` TEXT" for col in df.columns])
        cursor.execute(f"CREATE TABLE IF NOT EXISTS {table_name} ({cols});")

        logging.info(f"Inserting {len(df)} rows into {table_name}")
        placeholders = ", ".join(["%s"] * len(df.columns))
        insert_query = f"INSERT INTO {table_name} ({', '.join(df.columns)}) VALUES ({placeholders})"

        for _, row in df.iterrows():
            cursor.execute(insert_query, tuple(row))

        conn.commit()
        logging.info("Data load successful")
        print(f"✅ {len(df)} rows inserted into {table_name}")

    except Exception as e:
        logging.error(f"Error during loading: {e}")
        print("❌ Load Error:", e)
        sys.exit(1)

# ==============================
# MAIN ETL JOB
# ==============================
if __name__ == "__main__":
    logging.info("ETL Job Started")

    df = extract(CSV_FILE_PATH)
    df = transform(df)
    load(df, cursor, conn, TABLE_NAME)

    cursor.close()
    conn.close()
    logging.info("ETL Job Completed Successfully")
    print("✅ ETL Job Completed Successfully")


etl_job(Using sqlalchemy)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import logging
import os
import sys

# ============================
# CONFIGURATION SECTION
# ============================
DB_CONFIG = {
    'host': '127.0.0.1',       # or 'localhost'
    'user': 'root',
    'password': '',            # keep blank if no password
    'database': 'ecommerce_db',# target database
    'port': 3306
}

CSV_FILE_PATH = 'data/orders.csv'   # change this to your CSV file path
TABLE_NAME = 'orders'               # MySQL table name
LOG_FILE = 'etl_log.log'            # Log file name

# ============================
# LOGGING SETUP
# ============================
os.makedirs('log', exist_ok=True)
logging.basicConfig(
    filename=f'log/{LOG_FILE}',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ============================
# EXTRACT FUNCTION
# ============================
def extract(csv_path):
    try:
        logging.info(f"Extracting data from {csv_path}")
        df = pd.read_csv(csv_path)
        logging.info(f"Extraction successful. Rows: {len(df)} Columns: {list(df.columns)}")
        return df
    except Exception as e:
        logging.error(f"Error during extraction: {e}")
        sys.exit(1)

# ============================
# TRANSFORM FUNCTION
# ============================
def transform(df):
    try:
        logging.info("Starting data transformation")

        # Example transformations (customize as needed)
        df.columns = df.columns.str.lower().str.strip()   # normalize column names
        df.drop_duplicates(inplace=True)                  # remove duplicates
        df.fillna(value=None, inplace=True)               # handle NaN values

        logging.info("Transformation successful")
        return df
    except Exception as e:
        logging.error(f"Error during transformation: {e}")
        sys.exit(1)

# ============================
# LOAD FUNCTION
# ============================
def load(df, db_config, table_name):
    try:
        logging.info("Connecting to MySQL database")

        # Create MySQL engine using SQLAlchemy
        engine_url = f"mysql+pymysql://{db_config['user']}:{db_config['password']}@" \
                     f"{db_config['host']}:{db_config['port']}/{db_config['database']}"

        engine = create_engine(engine_url)

        logging.info(f"Loading data into table: {table_name}")
        df.to_sql(name=table_name, con=engine, if_exists='append', index=False)
        logging.info("Data loaded successfully")
    except Exception as e:
        logging.error(f"Error during loading: {e}")
        sys.exit(1)

# ============================
# MAIN ETL PIPELINE
# ============================
if __name__ == "__main__":
    logging.info("ETL Job Started")
    try:
        df_extracted = extract(CSV_FILE_PATH)
        df_transformed = transform(df_extracted)
        load(df_transformed, DB_CONFIG, TABLE_NAME)
        logging.info("ETL Job Completed Successfully")
    except Exception as e:
        logging.error(f"ETL job failed: {e}")
